In [1]:
import os

path = r"C:\Code\Ky5\SEG\Crawl\database\.env"
print("✅ File tồn tại:", os.path.exists(path))


✅ File tồn tại: True


In [ ]:
import os, re 
import mysql.connector
import pandas as pd
from rank_bm25 import BM25Okapi
from unidecode import unidecode
from dotenv import load_dotenv
from IPython.display import display, HTML

# Load .env để đọc thông tin MySQL
load_dotenv(r"C:\Code\Ky5\SEG\Crawl\database\.env")

MYSQL_HOST = os.getenv("MYSQL_HOST")
MYSQL_USER = os.getenv("MYSQL_USER")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD")
MYSQL_DB = os.getenv("MYSQL_DB")
MYSQL_TABLE = "law_chunks"

print("✅ Đã load cấu hình MySQL:")
print("Host:", MYSQL_HOST)
print("User:", MYSQL_USER)
print("DB:", MYSQL_DB)


✅ Đã load cấu hình MySQL:
Host: localhost
User: root
DB: VNLawsv3


In [6]:
# Kết nối MySQL
conn = mysql.connector.connect(
    host=MYSQL_HOST,
    user=MYSQL_USER,
    password=MYSQL_PASSWORD,
    database=MYSQL_DB
)

query = f"""
SELECT law_id, chunk_num, document_type, issuing_agency, issue_date,
       title, source_url, raw_title, category, chunk
FROM {MYSQL_TABLE}
"""

df = pd.read_sql(query, conn)
conn.close()

print(f"✅ Đã nạp {len(df)} đoạn văn (chunks)")
display(df.head(3))


C:\Users\ACER\AppData\Local\Temp\ipykernel_31356\507573455.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


✅ Đã nạp 374251 đoạn văn (chunks)


,law_id,chunk_num,document_type,issuing_agency,issue_date,title,source_url,raw_title,category,chunk
0,22714,1,NGHỊ ĐỊNH,BỘ TRƯỞNG BỘ QUỐC GIA GIÁO DỤC BAN HÀNH,1945-10-15,Nghị định năm 1945 về Hội đồng cố vấn học chín...,https://thuvienphapluat.vn/van-ban/Giao-duc/Ng...,Nghị định năm 1945 về Hội đồng cố vấn học chín...,Giáo dục,BỘ QUỐC GIA GIÁO DỤC VIỆT NAM DÂN CHỦ CỘNG HÒA...
1,22714,2,NGHỊ ĐỊNH,BỘ TRƯỞNG BỘ QUỐC GIA GIÁO DỤC BAN HÀNH,1945-10-15,Nghị định năm 1945 về Hội đồng cố vấn học chín...,https://thuvienphapluat.vn/van-ban/Giao-duc/Ng...,Nghị định năm 1945 về Hội đồng cố vấn học chín...,Giáo dục,TRƯỞNG BỘ QUỐC GIA GIÁO DỤC Chiếu chỉ Sắc lệnh...
2,22714,3,NGHỊ ĐỊNH,BỘ TRƯỞNG BỘ QUỐC GIA GIÁO DỤC BAN HÀNH,1945-10-15,Nghị định năm 1945 về Hội đồng cố vấn học chín...,https://thuvienphapluat.vn/van-ban/Giao-duc/Ng...,Nghị định năm 1945 về Hội đồng cố vấn học chín...,Giáo dục,đồng cố vấn học chính họp mỗi năm hai kỳ vào đ...


In [7]:
def clean_text(s: str) -> str:
    if not s:
        return ""
    s = s.replace("\r", "\n")
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"KHÔNGSỐ", "KHÔNG SỐ", s, flags=re.IGNORECASE)
    return s.strip()

def vi_tokenize(s: str):
    s = s.lower()
    s = re.sub(r"[^a-zà-ỹ0-9\s]", " ", s)
    tokens = s.split()
    expanded = []
    for t in tokens:
        expanded.append(t)
        t_ascii = unidecode(t)
        if t_ascii != t:
            expanded.append(t_ascii)
    return expanded

# Áp dụng
df["tokens"] = df["chunk"].apply(lambda x: vi_tokenize(clean_text(str(x))))


In [8]:
bm25 = BM25Okapi(df["tokens"].tolist())
print("✅ Đã xây dựng BM25 model thành công")


✅ Đã xây dựng BM25 model thành công


In [ ]:
def bm25_search(query: str, top_k=10, export_excel=True):
    q_tokens = vi_tokenize(query)
    scores = bm25.get_scores(q_tokens)

    df["score"] = scores
    results = df.sort_values("score", ascending=False).head(top_k).copy()

    # --- Tạo snippet highlight ---
    def make_snippet(text, query):
        text = clean_text(text)
        for qt in query.split():
            text = re.sub(f"({qt})", r"<b>\1</b>", text, flags=re.IGNORECASE)
        if len(text) > 300:
            text = text[:300] + "..."
        return text

    results["snippet"] = results["chunk"].apply(lambda t: make_snippet(t, query))

    # --- Hiển thị bảng kết quả trong Notebook ---
    display(HTML(
        results[[
            "law_id", "title", "category", "document_type", "issuing_agency", "score", "snippet"
        ]].to_html(escape=False, index=False)
    ))

    # --- Xuất ra Excel ---
    if export_excel:
        safe_query = re.sub(r"[\\/*?\"<>|]", "_", query)  # tránh ký tự lỗi trong tên file
        output_path = f"bm25_results_{safe_query}.xlsx"

        export_cols = [
            "law_id", "chunk_num", "title", "category", "document_type",
            "issuing_agency", "issue_date", "source_url", "score", "chunk"
        ]

        results.to_excel(output_path, index=False, columns=export_cols)
        print(f"✅ Đã lưu kết quả tìm kiếm vào file: {output_path}")

    return results


# --- Ví dụ ---
bm25_search("giáo dục tiểu học")


law_id,title,category,document_type,issuing_agency,score,snippet
38207,Nghị định 338-HĐBT năm 1991 thi hành Luật phổ cập giáo dục tiểu học do Hội đồng Bộ trưởng ban hành,Giáo dục,NGHỊ ĐỊNH,BỘ TRƯỞNG BAN HÀNH,37.816148,"tiểu học theo Luật định, phải chịu các hình thức xử phạt thích hợp. Điều 12. Bộ Giáo dục và Đào tạo có trách nhiệm: 1. Quyết định các chương chương trình và nội dung giáo dục tiểu học chung cho cả nước và riêng cho từng vùng quy định trong Điều..."
38065,Luật Phổ cập giáo dục tiểu học 1991,Giáo dục,LUẬT,QUỐC HỘI,37.669820,"chương trình, kế hoạch triển khai việc phổ cập giáo dục tiểu học, ban hành các văn bản pháp quy thuộc thẩm quyền; 2- Quy định mục mục tiêu và kế hoạch đào tạo của trường tiểu học, nội dung giáo dục tiểu học, quản lý việc biên soạn..."
55066,Quyết định 51/2007/QĐ-BGDĐT ban hành Điều lệ trường tiểu học do Bộ trưởng Bộ Giáo dục và Đào tạo ban hành,Giáo dục,QUYẾT ĐỊNH,BỘ TRƯỞNG BỘ GIÁO DỤC VÀ ĐÀO TẠO BAN HÀNH,37.626302,"này áp dụng cho trường tiểu học; lớp tiểu học trong trường phổ thông có nhiều cấp học và trường chuyên biệt; cơ sở giáo dục khác thực hiện chương chương trình giáo dục tiểu học; tổ chức, cá nhân tham gia hoạt động giáo</..."
38065,Luật Phổ cập giáo dục tiểu học 1991,Giáo dục,LUẬT,QUỐC HỘI,37.559668,"tuổi hoặc học vượt lớp khi cơ quan quản lý giáo dục có thẩm quyền cho phép. Điều 9 Học sinh phải được học tập và rèn luyện theo chương chương trình, nội dung giáo dục tiểu học do Nhà nước quy định; được tham gia các hoạt động của Đội thi..."
46532,Quyết định 22/2000/QĐ-BGDĐT về Điều lệ Trường tiểu học do Bộ trưởng Bộ Giáo dục và Đào tạo ban hành,Giáo dục,QUYẾT ĐỊNH,BỘ TRƯỞNG BỘ GIÁO DỤC VÀ ĐÀO TẠO BAN HÀNH,37.405394,ĐIỀU LỆ TRƯỜNG TIỂU HỌC (Ban hành kèm theo Quyết định số: 22 /2000/QĐ-BGD&ĐT ngày 11 tháng 7 năm 2000 của Bộ trưởng Bộ Giáo dục và Đào tạo) Chương Chương 1 NHỮNG QUY ĐỊNH CHUNG Điều 1. Phạm vi điều chỉnh Điều lệ này quy định về tổ chức và hoạt động của trường tiểu ...
46532,Quyết định 22/2000/QĐ-BGDĐT về Điều lệ Trường tiểu học do Bộ trưởng Bộ Giáo dục và Đào tạo ban hành,Giáo dục,QUYẾT ĐỊNH,BỘ TRƯỞNG BỘ GIÁO DỤC VÀ ĐÀO TẠO BAN HÀNH,37.405394,ĐIỀU LỆ TRƯỜNG TIỂU HỌC (Ban hành kèm theo Quyết định số: 22 /2000/QĐ-BGD&ĐT ngày 11 tháng 7 năm 2000 của Bộ trưởng Bộ Giáo dục và Đào tạo) Chương Chương 1 NHỮNG QUY ĐỊNH CHUNG Điều 1. Phạm vi điều chỉnh Điều lệ này quy định về tổ chức và hoạt động của trường tiểu ...
38065,Luật Phổ cập giáo dục tiểu học 1991,Giáo dục,LUẬT,QUỐC HỘI,37.308859,"quyền địa phương thực hiện các quy định tại Điều này. Điều 24 Nội dung thanh tra giáo dục tiểu học bao gồm: 1- Thanh tra việc thực hiện chương chương trình - mục mục tiêu, kế hoạch phổ cập giáo dục tiểu học của các địa phương, trường, lớp ti..."
55066,Quyết định 51/2007/QĐ-BGDĐT ban hành Điều lệ trường tiểu học do Bộ trưởng Bộ Giáo dục và Đào tạo ban hành,Giáo dục,QUYẾT ĐỊNH,BỘ TRƯỞNG BỘ GIÁO DỤC VÀ ĐÀO TẠO BAN HÀNH,36.979735,điều kiện theo quy định của Bộ trưởng Bộ Giáo dục và Đào tạo thì được Hiệu trưởng trường tiểu học xác nhận trong học bạ việc hoàn thành chương chương trình tiểu học. 3. Đối với cơ sở giáo dục khác thực hiện chương chương trình giáo</b...
143853,"Quyết định 4499/QĐ-BGDĐT năm 2007 về Quy định tạm thời chức năng, nhiệm vụ của tổ chức giúp Bộ trưởng thực hiện chức năng quản lý nhà nước thuộc Bộ Giáo dục và Đào tạo",Bộ máy hành chính,QUYẾT ĐỊNH,BỘ TRƯỞNG THỰC HIỆN CHỨC NĂNG QUẢN LÝ NHÀ NƯỚC THUỘC BỘ GIÁO DỤC VÀ ĐÀO TẠO,36.955467,"Trung ương trong việc thực hiện chức năng quản lý nhà nước về giáo dục tiểu học theo phân cấp của Chính phủ. 4- Giúp Bộ trưởng quản lý chương chương trình, nội dung, kế hoạch dạy học của mọi loại hình trường, lớp thuộc giáo dục tiểu học;..."
54699,"Quyết định 4494/QĐ-BGDĐT năm 2007 quy định tạm thời chức năng, nhiệm vụ của các tổ chức giúp Bộ trưởng thực hiện chức năng quản lý nhà nước thuộc Bộ Giáo dục và Đào tạo do Bộ trưởng Bộ Giáo dục và Đào tạo ban hành",Bộ máy hành chính,QUYẾT ĐỊNH,BỘ TRƯỞNG THỰC HIỆN CHỨC NĂNG QUẢN LÝ NHÀ NƯỚC THUỘC BỘ GIÁO DỤC VÀ Đ

✅ Đã lưu kết quả tìm kiếm vào file: bm25_results_giáo dục tiểu học.xlsx


,law_id,chunk_num,document_type,issuing_agency,issue_date,title,source_url,raw_title,category,chunk,tokens,score,snippet
102280,38207,6,NGHỊ ĐỊNH,BỘ TRƯỞNG BAN HÀNH,1991-10-26,Nghị định 338-HĐBT năm 1991 thi hành Luật phổ ...,https://thuvienphapluat.vn/van-ban/Giao-duc/Ng...,Nghị định 338-HĐBT năm 1991 thi hành Luật phổ ...,Giáo dục,"tiểu học theo Luật định, phải chịu các hình th...","[tiểu, tieu, học, hoc, theo, luật, luat, định,...",37.816148,"<b>tiểu</b> <b>học</b> theo Luật định, phải ch..."
101338,38065,9,LUẬT,QUỐC HỘI,1991-08-12,Luật Phổ cập giáo dục tiểu học 1991,https://thuvienphapluat.vn/van-ban/Giao-duc/Lu...,Luật Phổ cập giáo dục tiểu học 1991,Giáo dục,"chương trình, kế hoạch triển khai việc phổ cập...","[chương, chuong, trình, trinh, kế, ke, hoạch, ...",37.669820,"chương trình, kế hoạch triển khai việc phổ cập..."
357891,55066,5,QUYẾT ĐỊNH,BỘ TRƯỞNG BỘ GIÁO DỤC VÀ ĐÀO TẠO BAN HÀNH,2007-08-31,Quyết định 51/2007/QĐ-BGDĐT ban hành Điều lệ t...,https://thuvienphapluat.vn/van-ban/Giao-duc/Qu...,Quyết định 51/2007/QĐ-BGDĐT ban hành Điều lệ t...,Giáo dục,này áp dụng cho trường tiểu học; lớp tiểu học ...,"[này, nay, áp, ap, dụng, dung, cho, trường, tr...",37.626302,này áp dụng cho trường <b>tiểu</b> <b>học</b>;...
101334,38065,5,LUẬT,QUỐC HỘI,1991-08-12,Luật Phổ cập giáo dục tiểu học 1991,https://thuvienphapluat.vn/van-ban/Giao-duc/Lu...,Luật Phổ cập giáo dục tiểu học 1991,Giáo dục,tuổi hoặc học vượt lớp khi cơ quan quản lý giá...,"[tuổi, tuoi, hoặc, hoac, học, hoc, vượt, vuot,...",37.559668,tuổi hoặc <b>học</b> vượt lớp khi cơ quan quản...
220598,46532,3,QUYẾT ĐỊNH,BỘ TRƯỞNG BỘ GIÁO DỤC VÀ ĐÀO TẠO BAN HÀNH,2000-07-11,Quyết định 22/2000/QĐ-BGDĐT về Điều lệ Trường ...,https://thuvienphapluat.vn/van-ban/Giao-duc/Qu...,Quyết định 22/2000/QĐ-BGDĐT về Điều lệ Trường ...,Giáo dục,ĐIỀU LỆ TRƯỜNG TIỂU HỌC (Ban hành kèm theo Quy...,"[điều, dieu, lệ, le, trường, truong, tiểu, tie...",37.405394,ĐIỀU LỆ TRƯỜNG <b>TIỂU</b> <b>HỌC</b> (Ban hàn...
220620,46532,25,QUYẾT ĐỊNH,BỘ TRƯỞNG BỘ GIÁO DỤC VÀ ĐÀO TẠO BAN HÀNH,2000-07-11,Quyết định 22/2000/QĐ-BGDĐT về Điều lệ Trường ...,https://thuvienphapluat.vn/van-ban/Giao-duc/Qu...,Quyết định 22/2000/QĐ-BGDĐT về Điều lệ Trường ...,Giáo dục,ĐIỀU LỆ TRƯỜNG TIỂU HỌC (Ban hành kèm theo Quy...,"[điều, dieu, lệ, le, trường, truong, tiểu, tie...",37.405394,ĐIỀU LỆ TRƯỜNG <b>TIỂU</b> <b>HỌC</b> (Ban hàn...
101341,38065,12,LUẬT,QUỐC HỘI,1991-08-12,Luật Phổ cập giáo dục tiểu học 1991,https://thuvienphapluat.vn/van-ban/Giao-duc/Lu...,Luật Phổ cập giáo dục tiểu học 1991,Giáo dục,quyền địa phương thực hiện các quy định tại Đi...,"[quyền, quyen, địa, dia, phương, phuong, thực,...",37.308859,quyền địa phương thực hiện các quy định tại Đi...
357914,55066,28,QUYẾT ĐỊNH,BỘ TRƯỞNG BỘ GIÁO DỤC VÀ ĐÀO TẠO BAN HÀNH,2007-08-31,Quyết định 51/2007/QĐ-BGDĐT ban hành Điều lệ t...,https://thuvienphapluat.vn/van-ban/Giao-duc/Qu...,Quyết định 51/2007/QĐ-BGDĐT ban hành Điều lệ t...,Giáo dục,điều kiện theo quy định của Bộ trưởng Bộ Giáo ...,"[điều, dieu, kiện, kien, theo, quy, định, dinh...",36.979735,điều kiện theo quy định của Bộ trưởng Bộ <b>Gi...
354647,143853,9,QUYẾT ĐỊNH,BỘ TRƯỞNG THỰC HIỆN CHỨC NĂNG QUẢN LÝ NHÀ NƯỚC...,2007-08-23,Quyết định 4499/QĐ-BGDĐT năm 2007 về Quy định ...,https://thuvienphapluat.vn/van-ban/Bo-may-hanh...,Quyết định 4499/QĐ-BGDĐT năm 2007 về Quy định ...,Bộ máy hành chính,Trung ương trong việc thực hiện chức năng quản...,"[trung, ương, uong, trong, việc, viec, thực, t...",36.955467,Trung ương trong việc thực hiện chức năng quản...
355190,54699,11,QUYẾT ĐỊNH,BỘ TRƯỞNG THỰC HIỆN CHỨC NĂNG QUẢN LÝ NHÀ NƯỚC...,2007-08-23,Quyết định 4494/QĐ-BGDĐT năm 2007 quy định tạm...,https://thuvienphapluat.vn/van-ban/Bo-may-hanh...,Quyết định 4494/QĐ-BGDĐT năm 2007 quy định tạm...,Bộ máy hành chính,Trung ương trong việc thực hiện chức năng quản...,"[trung, ương, uong, trong, việc, viec, thực, t...",36.955467,Trung ương trong việc thực hiện chức năng quản...


In [13]:
def search_grouped(query, top_k=5):
    q_tokens = vi_tokenize(query)
    scores = bm25.get_scores(q_tokens)
    df["score"] = scores
    grouped = df.groupby("law_id").agg({
        "score": "max",
        "title": "first",
        "category": "first",
        "document_type": "first",
        "issuing_agency": "first",
        "source_url": "first"
    }).sort_values("score", ascending=False).head(top_k)

    display(HTML(grouped.to_html(escape=False)))
    return grouped


In [17]:
results = search_grouped("giáo dục tiểu học")
# Hiển thị toàn văn bản top-1


,score,title,category,document_type,issuing_agency,source_url
law_id,,,,,,
38207,37.816148,Nghị định 338-HĐBT năm 1991 thi hành Luật phổ cập giáo dục tiểu học do Hội đồng Bộ trưởng ban hành,Giáo dục,NGHỊ ĐỊNH,BỘ TRƯỞNG BAN HÀNH,https://thuvienphapluat.vn/van-ban/Giao-duc/Nghi-dinh-338-HDBT-thi-hanh-Luat-pho-cap-giao-duc-tieu-hoc-38207.aspx
38065,37.669820,Luật Phổ cập giáo dục tiểu học 1991,Giáo dục,LUẬT,QUỐC HỘI,https://thuvienphapluat.vn/van-ban/Giao-duc/Luat-Pho-cap-giao-duc-tieu-hoc-1991-56-LCT-HDNN8-38065.aspx
55066,37.626302,Quyết định 51/2007/QĐ-BGDĐT ban hành Điều lệ trường tiểu học do Bộ trưởng Bộ Giáo dục và Đào tạo ban hành,Giáo dục,QUYẾT ĐỊNH,BỘ TRƯỞNG BỘ GIÁO DỤC VÀ ĐÀO TẠO BAN HÀNH,https://thuvienphapluat.vn/van-ban/Giao-duc/Quyet-dinh-51-2007-QD-BGDDT-Dieu-le-truong-tieu-hoc-55066.aspx
46532,37.405394,Quyết định 22/2000/QĐ-BGDĐT về Điều lệ Trường tiểu học do Bộ trưởng Bộ Giáo dục và Đào tạo ban hành,Giáo dục,QUYẾT ĐỊNH,BỘ TRƯỞNG BỘ GIÁO DỤC VÀ ĐÀO TẠO BAN HÀNH,https://thuvienphapluat.vn/van-ban/Giao-duc/Quyet-dinh-22-2000-QD-BGDDT-Dieu-le-Truong-tieu-hoc-46532.aspx
143853,36.955467,"Quyết định 4499/QĐ-BGDĐT năm 2007 về Quy định tạm thời chức năng, nhiệm vụ của tổ chức giúp Bộ trưởng thực hiện chức năng quản lý nhà nước thuộc Bộ Giáo dục và Đào tạo",Bộ máy hành chính,QUYẾT ĐỊNH,BỘ TRƯỞNG THỰC HIỆN CHỨC NĂNG QUẢN LÝ NHÀ NƯỚC THUỘC BỘ GIÁO DỤC VÀ ĐÀO TẠO,https://thuvienphapluat.vn/van-ban/Bo-may-hanh-chinh/Quyet-dinh-4499-QD-BGDDT-nam-2007-Quy-dinh-tam-thoi-chuc-nang-nhiem-vu-143853.aspx
